# Deep Dive: Build the Cat Health Agentic RAG Loop From Scratch

The main Session 2 notebook builds agentic RAG with **LangChain**, **LangGraph**, **OpenAI models**, and **Qdrant**. This optional reference notebook rebuilds the same application with the agent loop exposed.

We will use Python standard-library code for HTTP requests, provider adapters, tool definitions, tool execution, loop control, callbacks, Markdown loading, embeddings, cosine similarity, and brute-force vector search.

The application-facing abstractions intentionally mirror current **Vercel AI SDK Core** concepts:

```text
tool() + generate_text() + step_count_is() + ToolLoopAgent
```

The goal is not to create a production-ready SDK. The goal is to understand what agent libraries do for us.

```text
question -> model -> optional tool call -> execute tool -> return result -> model -> answer
```


## Table of Contents

- 1. Environment Setup
- 2. Raw HTTP Transport
- 3. AI SDK-Inspired Core Primitives
- 4. OpenAI Provider Adapter
- 5. Build the Retriever From Scratch
- 6. Turn Retrieval Into a Tool
- 7. Expose the Multi-Step Agent Loop
- 8. Create a Reusable `ToolLoopAgent`
- 9. Change Loop Behavior with `prepare_step`
- Production Library Responsibilities


## 1. Environment Setup

From the `02_Agentic_RAG_LangGraph_LangChain` folder, install dependencies with uv:

```bash
uv sync
```

Then open this notebook in Cursor or VS Code and select the Python/Jupyter environment created by uv.

This deep dive uses only the Python standard library. The existing Session 2 environment is still convenient because it already includes Jupyter.


### Imports

Everything below is implemented with Python standard-library modules.


In [ ]:
from collections.abc import Callable
from dataclasses import dataclass, field
from getpass import getpass
from math import sqrt
from pathlib import Path
from typing import Any, Protocol
from urllib.error import HTTPError, URLError
from urllib.request import Request, urlopen
import json
import os


### OpenAI API Key and Model Defaults

The embedding and language-model adapters call OpenAI directly. If `OPENAI_API_KEY` is not already set, this cell asks for it securely.

The defaults match the main Session 2 notebook. You can override them with environment variables before running the notebook.


In [ ]:
if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("OpenAI API Key: ")

openai_base_url = os.environ.get("OPENAI_BASE_URL", "https://api.openai.com/v1").rstrip("/")
embedding_model_id = os.environ.get("AIM_EMBEDDING_MODEL", "text-embedding-3-small")
chat_model_id = os.environ.get("AIM_CHAT_MODEL", "gpt-5.4-mini")

print(f"Embedding model: {embedding_model_id}")
print(f"Chat model: {chat_model_id}")


## 2. Raw HTTP Transport

Provider SDKs handle request construction, authentication, JSON serialization, JSON parsing, API errors, retries, and more. Our `_post_json()` helper makes the basic request boundary visible.

It intentionally does **not** implement retries, exponential backoff, streaming, telemetry, or concurrent requests.


In [ ]:
def _post_json(
    *,
    path: str,
    payload: dict[str, Any],
    api_key: str,
    base_url: str,
    timeout: int = 60,
) -> dict[str, Any]:
    """POST a JSON payload and return the decoded JSON response."""
    if not api_key:
        raise ValueError("An OpenAI API key is required.")

    request = Request(
        url=f"{base_url}{path}",
        data=json.dumps(payload).encode("utf-8"),
        headers={
            "Authorization": f"Bearer {api_key}",
            "Content-Type": "application/json",
        },
        method="POST",
    )

    try:
        with urlopen(request, timeout=timeout) as response:
            response_body = response.read().decode("utf-8")
    except HTTPError as exc:
        error_body = exc.read().decode("utf-8", errors="replace")
        try:
            error_payload = json.loads(error_body)
            error_message = error_payload.get("error", {}).get("message", error_body)
        except json.JSONDecodeError:
            error_message = error_body
        raise RuntimeError(
            f"OpenAI API request failed with HTTP {exc.code}: {error_message}"
        ) from exc
    except URLError as exc:
        raise RuntimeError(f"OpenAI API request could not connect: {exc.reason}") from exc

    try:
        return json.loads(response_body)
    except json.JSONDecodeError as exc:
        raise RuntimeError("OpenAI API returned a response that was not valid JSON.") from exc


## 3. AI SDK-Inspired Core Primitives

An agent loop needs a few small contracts:

- A `Tool` describes an action, its JSON input schema, and the function that executes it.
- A `ToolCall` is a model request to run a named tool with structured input.
- A `ToolResult` is application code returning the result of that action.
- A `StepResult` records one model step and any tool calls executed after it.
- A `GenerateTextResult` returns the final text, every step, and total usage.
- A `LanguageModel` adapter turns provider-neutral requests into provider API calls.

The provider continuation value is intentionally opaque. Our OpenAI adapter uses it to continue a Responses API conversation after tool execution, while the core loop does not need to know its shape.


In [ ]:
ToolExecutor = Callable[[dict[str, Any]], Any]


@dataclass(frozen=True)
class Tool:
    description: str
    input_schema: dict[str, Any]
    execute: ToolExecutor


@dataclass(frozen=True)
class ToolCall:
    tool_call_id: str
    tool_name: str
    input: dict[str, Any]


@dataclass(frozen=True)
class ToolResult:
    tool_call_id: str
    tool_name: str
    input: dict[str, Any]
    output: Any
    is_error: bool = False


@dataclass
class ModelStepResult:
    text: str
    tool_calls: list[ToolCall]
    usage: dict[str, Any]
    finish_reason: str
    continuation: Any
    raw_response: dict[str, Any]


@dataclass
class StepResult:
    step_number: int
    text: str
    tool_calls: list[ToolCall]
    tool_results: list[ToolResult]
    usage: dict[str, Any]
    finish_reason: str
    raw_response: dict[str, Any]


@dataclass
class GenerateTextResult:
    text: str
    steps: list[StepResult]
    total_usage: dict[str, Any]


@dataclass(frozen=True)
class PrepareStepOptions:
    step_number: int
    steps: list[StepResult]
    active_tools: list[str]


@dataclass(frozen=True)
class PrepareStepResult:
    active_tools: list[str] | None = None


StopCondition = Callable[[list[StepResult]], bool]
OnStepFinish = Callable[[StepResult], None]
PrepareStep = Callable[[PrepareStepOptions], PrepareStepResult | None]


class LanguageModel(Protocol):
    def do_generate(
        self,
        *,
        instructions: str,
        prompt: str | None,
        tools: dict[str, Tool],
        continuation: Any,
        tool_results: list[ToolResult],
    ) -> ModelStepResult:
        ...


### `tool()`, Input Validation, and Stop Conditions

AI SDK tools use a description, an input schema, and an execute function. The description helps the model decide when to call the tool. The JSON schema tells the model what arguments to produce and lets the application validate them before execution.

Our validator intentionally supports only the small JSON-schema subset needed by this notebook.

`step_count_is()` returns a stop-condition function. The multi-step loop will evaluate it after a step that produced tool results.


In [ ]:
def tool(
    *,
    description: str,
    input_schema: dict[str, Any],
    execute: ToolExecutor,
) -> Tool:
    """Create a tool definition with a JSON input schema and executor."""
    if not description.strip():
        raise ValueError("A tool description is required.")
    if input_schema.get("type") != "object":
        raise ValueError("This teaching implementation expects an object input schema.")
    return Tool(description=description, input_schema=input_schema, execute=execute)


def _matches_json_type(value: Any, expected_type: str) -> bool:
    type_checks = {
        "string": lambda item: isinstance(item, str),
        "number": lambda item: isinstance(item, (int, float)) and not isinstance(item, bool),
        "integer": lambda item: isinstance(item, int) and not isinstance(item, bool),
        "boolean": lambda item: isinstance(item, bool),
        "object": lambda item: isinstance(item, dict),
        "array": lambda item: isinstance(item, list),
        "null": lambda item: item is None,
    }
    if expected_type not in type_checks:
        raise ValueError(f"Unsupported JSON schema type: {expected_type}")
    return type_checks[expected_type](value)


def _validate_tool_input(tool_definition: Tool, tool_input: dict[str, Any]) -> None:
    schema = tool_definition.input_schema
    properties = schema.get("properties", {})
    required = schema.get("required", [])

    missing = [name for name in required if name not in tool_input]
    if missing:
        raise ValueError(f"Missing required tool input fields: {missing}")

    if schema.get("additionalProperties") is False:
        unexpected = [name for name in tool_input if name not in properties]
        if unexpected:
            raise ValueError(f"Unexpected tool input fields: {unexpected}")

    for name, value in tool_input.items():
        expected_type = properties.get(name, {}).get("type")
        if expected_type and not _matches_json_type(value, expected_type):
            raise ValueError(
                f"Tool input field {name!r} must have JSON type {expected_type!r}."
            )


def step_count_is(count: int) -> StopCondition:
    """Stop after the requested number of model steps."""
    if count <= 0:
        raise ValueError("Step count must be greater than zero.")
    return lambda steps: len(steps) >= count


def _execute_tool_call(
    tool_call: ToolCall,
    *,
    active_tools: dict[str, Tool],
) -> ToolResult:
    """Validate and execute one model-requested tool call."""
    tool_definition = active_tools.get(tool_call.tool_name)
    if tool_definition is None:
        return ToolResult(
            tool_call_id=tool_call.tool_call_id,
            tool_name=tool_call.tool_name,
            input=tool_call.input,
            output={"error": f"Tool {tool_call.tool_name!r} is not available."},
            is_error=True,
        )

    try:
        _validate_tool_input(tool_definition, tool_call.input)
        output = tool_definition.execute(tool_call.input)
        return ToolResult(
            tool_call_id=tool_call.tool_call_id,
            tool_name=tool_call.tool_name,
            input=tool_call.input,
            output=output,
        )
    except Exception as exc:
        return ToolResult(
            tool_call_id=tool_call.tool_call_id,
            tool_name=tool_call.tool_name,
            input=tool_call.input,
            output={"error": str(exc)},
            is_error=True,
        )


## 4. OpenAI Provider Adapter

The application-facing agent loop should not need to understand OpenAI request or response shapes.

The adapter below:

1. Converts our named `Tool` mapping into OpenAI function-tool definitions.
2. Calls the Responses API.
3. Parses function-call output items into our `ToolCall` type.
4. Returns an opaque continuation ID so the next step can send tool results back to the same provider conversation.

This is one of the major jobs a provider abstraction performs.


In [ ]:
@dataclass
class EmbedManyResult:
    values: list[str]
    embeddings: list[list[float]]
    usage: dict[str, Any]
    raw_response: dict[str, Any]


class EmbeddingModel(Protocol):
    def do_embed(self, values: list[str]) -> EmbedManyResult:
        ...


def embed_many(*, model: EmbeddingModel, values: list[str]) -> EmbedManyResult:
    """Embed several values with a provider-neutral embedding model."""
    return model.do_embed(values)


def embed(*, model: EmbeddingModel, value: str) -> list[float]:
    """Embed one value using the same batch primitive."""
    return embed_many(model=model, values=[value]).embeddings[0]


def _extract_output_text(response: dict[str, Any]) -> str:
    text_parts = []
    for output_item in response.get("output", []):
        for content_part in output_item.get("content", []):
            if content_part.get("type") == "output_text":
                text_parts.append(content_part.get("text", ""))
    return "".join(text_parts)


def _openai_tool_specs(tools: dict[str, Tool]) -> list[dict[str, Any]]:
    return [
        {
            "type": "function",
            "name": name,
            "description": definition.description,
            "parameters": definition.input_schema,
            "strict": True,
        }
        for name, definition in tools.items()
    ]


In [ ]:
@dataclass(frozen=True)
class OpenAIEmbeddingModel:
    model_id: str
    api_key: str
    base_url: str

    def do_embed(self, values: list[str]) -> EmbedManyResult:
        if not values:
            raise ValueError("At least one value is required for embedding.")
        if any(not value.strip() for value in values):
            raise ValueError("Embedding values must not be empty strings.")

        raw_response = _post_json(
            path="/embeddings",
            payload={
                "model": self.model_id,
                "input": values,
                "encoding_format": "float",
            },
            api_key=self.api_key,
            base_url=self.base_url,
        )

        ordered_data = sorted(raw_response.get("data", []), key=lambda item: item["index"])
        embeddings = [item["embedding"] for item in ordered_data]
        if len(embeddings) != len(values):
            raise RuntimeError("OpenAI API returned an unexpected number of embeddings.")

        return EmbedManyResult(
            values=list(values),
            embeddings=embeddings,
            usage=raw_response.get("usage", {}),
            raw_response=raw_response,
        )


@dataclass(frozen=True)
class OpenAILanguageModel:
    model_id: str
    api_key: str
    base_url: str

    def do_generate(
        self,
        *,
        instructions: str,
        prompt: str | None,
        tools: dict[str, Tool],
        continuation: str | None,
        tool_results: list[ToolResult],
    ) -> ModelStepResult:
        if continuation is None:
            if not prompt or not prompt.strip():
                raise ValueError("The first model step requires a non-empty prompt.")
            model_input: str | list[dict[str, Any]] = prompt
        else:
            if not tool_results:
                raise ValueError("A continuation step requires tool results.")
            model_input = [
                {
                    "type": "function_call_output",
                    "call_id": result.tool_call_id,
                    "output": json.dumps(
                        {"result": result.output, "is_error": result.is_error},
                        default=str,
                    ),
                }
                for result in tool_results
            ]

        payload: dict[str, Any] = {
            "model": self.model_id,
            "instructions": instructions,
            "input": model_input,
            "store": True,
        }
        if tools:
            payload["tools"] = _openai_tool_specs(tools)
        if continuation is not None:
            payload["previous_response_id"] = continuation

        raw_response = _post_json(
            path="/responses",
            payload=payload,
            api_key=self.api_key,
            base_url=self.base_url,
        )

        tool_calls = []
        for item in raw_response.get("output", []):
            if item.get("type") != "function_call":
                continue

            raw_arguments = item.get("arguments", "{}")
            try:
                parsed_arguments = json.loads(raw_arguments)
            except json.JSONDecodeError:
                parsed_arguments = {"_invalid_json": raw_arguments}
            if not isinstance(parsed_arguments, dict):
                parsed_arguments = {"value": parsed_arguments}

            tool_calls.append(
                ToolCall(
                    tool_call_id=item["call_id"],
                    tool_name=item["name"],
                    input=parsed_arguments,
                )
            )

        return ModelStepResult(
            text=_extract_output_text(raw_response),
            tool_calls=tool_calls,
            usage=raw_response.get("usage", {}),
            finish_reason="tool-calls" if tool_calls else raw_response.get("status", "stop"),
            continuation=raw_response.get("id"),
            raw_response=raw_response,
        )


@dataclass(frozen=True)
class OpenAIProvider:
    api_key: str
    base_url: str = "https://api.openai.com/v1"

    def embedding_model(self, model_id: str) -> OpenAIEmbeddingModel:
        return OpenAIEmbeddingModel(
            model_id=model_id,
            api_key=self.api_key,
            base_url=self.base_url,
        )

    def language_model(self, model_id: str) -> OpenAILanguageModel:
        return OpenAILanguageModel(
            model_id=model_id,
            api_key=self.api_key,
            base_url=self.base_url,
        )


In [ ]:
provider = OpenAIProvider(
    api_key=os.environ["OPENAI_API_KEY"],
    base_url=openai_base_url,
)
embedding_model = provider.embedding_model(embedding_model_id)
language_model = provider.language_model(chat_model_id)


## 5. Build the Retriever From Scratch

The main notebook uses LangChain document utilities and Qdrant. Here we expose those responsibilities again so the retrieval tool is independently runnable:

1. Load the Markdown corpus.
2. Split it into heading-based chunks.
3. Embed each chunk.
4. Store vectors and documents together.
5. Embed each search query and rank chunks with cosine similarity.

The retrieval implementation is not the agent loop. It is an application capability that we will wrap as a tool.


In [ ]:
@dataclass
class Document:
    page_content: str
    metadata: dict[str, Any]


def load_markdown(path: Path) -> Document:
    if not path.exists():
        raise FileNotFoundError(
            f"Expected the cat health corpus at: {path.resolve()}\\n"
            "Run this notebook from the 02_Agentic_RAG_LangGraph_LangChain folder."
        )
    return Document(
        page_content=path.read_text(encoding="utf-8"),
        metadata={"source": path.name, "document_type": "cat_health_guidelines"},
    )


def split_markdown_sections(document: Document) -> list[Document]:
    """Split a Markdown document whenever a level-two heading begins."""
    chunks: list[Document] = []
    current_heading = "Introduction"
    current_lines: list[str] = []

    def append_chunk() -> None:
        text = "\\n".join(current_lines).strip()
        if not text:
            return
        chunks.append(
            Document(
                page_content=text,
                metadata={
                    **document.metadata,
                    "section": current_heading,
                    "chunk_id": f"chunk-{len(chunks):02d}",
                },
            )
        )

    for line in document.page_content.splitlines():
        if line.startswith("## "):
            append_chunk()
            current_heading = line.removeprefix("## ").strip()
            current_lines = [line]
        else:
            current_lines.append(line)

    append_chunk()
    return chunks


corpus_path = Path("data/cat_health_guidelines.md")
corpus_document = load_markdown(corpus_path)
splits = split_markdown_sections(corpus_document)

print(f"Loaded {corpus_path.name} and created {len(splits)} heading-based chunks.")
for split in splits[:3]:
    print(split.metadata["chunk_id"], "-", split.metadata["section"])


In [ ]:
def cosine_similarity(vector_a: list[float], vector_b: list[float]) -> float:
    if not vector_a or not vector_b:
        raise ValueError("Cosine similarity requires two non-empty vectors.")
    if len(vector_a) != len(vector_b):
        raise ValueError("Cosine similarity requires vectors with matching dimensions.")

    dot_product = sum(a * b for a, b in zip(vector_a, vector_b))
    length_a = sqrt(sum(a * a for a in vector_a))
    length_b = sqrt(sum(b * b for b in vector_b))
    if length_a == 0 or length_b == 0:
        raise ValueError("Cosine similarity is undefined for zero-length vectors.")
    return dot_product / (length_a * length_b)


@dataclass
class StoredVector:
    document: Document
    embedding: list[float]


class InMemoryVectorStore:
    def __init__(self, embedding_model: EmbeddingModel):
        self.embedding_model = embedding_model
        self.records: list[StoredVector] = []

    def add_documents(self, documents: list[Document]) -> None:
        result = embed_many(
            model=self.embedding_model,
            values=[document.page_content for document in documents],
        )
        self.records.extend(
            StoredVector(document=document, embedding=embedding)
            for document, embedding in zip(documents, result.embeddings)
        )

    def similarity_search_with_score(
        self,
        query: str,
        *,
        k: int,
    ) -> list[tuple[Document, float]]:
        if not query.strip():
            raise ValueError("A non-empty retrieval query is required.")
        if k <= 0:
            raise ValueError("k must be greater than zero.")
        if not self.records:
            raise ValueError("Add documents before searching the vector store.")

        query_embedding = embed(model=self.embedding_model, value=query)
        scored_documents = [
            (record.document, cosine_similarity(query_embedding, record.embedding))
            for record in self.records
        ]
        return sorted(scored_documents, key=lambda item: item[1], reverse=True)[:k]


vector_store = InMemoryVectorStore(embedding_model)
vector_store.add_documents(splits)
retrieval_k = 4

print(f"Stored {len(vector_store.records)} vectors in memory.")


Inspect retrieval directly before giving it to the model. A tool does not improve a weak retriever; it only gives the model the option to call it.


In [ ]:
retrieval_query = "What urinary signs suggest a cat needs urgent veterinary care?"

for document, score in vector_store.similarity_search_with_score(
    retrieval_query,
    k=retrieval_k,
):
    print(
        f"{document.metadata['chunk_id']} | "
        f"{document.metadata['section']} | score={score:.3f}"
    )
    print(document.page_content[:500])
    print("-" * 100)


## 6. Turn Retrieval Into a Tool

Retrieval becomes agentic when it is an available action instead of a mandatory pre-step.

The tool contract matters:

- The **name** says what action exists.
- The **description** helps the model decide when to use it.
- The **input schema** defines and validates the query.
- The **execute function** runs application code and returns source-labeled context.

Notice that `tool()` does not know anything about vector databases. It only describes a callable capability.


In [ ]:
def retrieve_cat_health_guidelines(tool_input: dict[str, Any]) -> dict[str, Any]:
    query = tool_input["query"]
    scored_documents = vector_store.similarity_search_with_score(query, k=retrieval_k)

    sources = []
    for index, (document, score) in enumerate(scored_documents, start=1):
        sources.append(
            {
                "label": f"Source {index}",
                "source": document.metadata["source"],
                "section": document.metadata["section"],
                "chunk_id": document.metadata["chunk_id"],
                "score": round(score, 3),
                "content": document.page_content,
            }
        )

    return {"query": query, "sources": sources}


retriever_tool = tool(
    description=(
        "Search the cat health guideline corpus for relevant context about cat preventive "
        "care, nutrition, hydration, vaccines, parasites, dental health, urinary warning "
        "signs, emergencies, senior cats, stress, behavior, and safe home monitoring."
    ),
    input_schema={
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "A focused semantic search query for the cat health corpus.",
            }
        },
        "required": ["query"],
        "additionalProperties": False,
    },
    execute=retrieve_cat_health_guidelines,
)

tools = {"retrieve_cat_health_guidelines": retriever_tool}


Call the tool directly once. This is the context the model will see after it asks to retrieve.


In [ ]:
direct_tool_result = retriever_tool.execute(
    {"query": "What urinary signs suggest a cat needs urgent veterinary care?"}
)
print(json.dumps(direct_tool_result, indent=2)[:3000])


## 7. Expose the Multi-Step Agent Loop

This is the heart of the notebook.

AI SDK's `generateText()` can execute tools and continue across multiple model steps when given a `stopWhen` condition. Our `generate_text()` makes that loop visible:

```text
call model
  -> if no tool call: return final answer
  -> if tool call: validate and execute tool
  -> record the step and tool result
  -> stop if stop_when says so
  -> otherwise send tool result back to model and loop
```

By default, `generate_text()` uses `step_count_is(1)`, matching a single-step generation. Passing a larger stop condition enables the model-tools loop.


In [ ]:
def _sum_usage(steps: list[StepResult]) -> dict[str, Any]:
    totals: dict[str, Any] = {}
    for step in steps:
        for key, value in step.usage.items():
            if isinstance(value, (int, float)) and not isinstance(value, bool):
                totals[key] = totals.get(key, 0) + value
    return totals


def generate_text(
    *,
    model: LanguageModel,
    instructions: str,
    prompt: str,
    tools: dict[str, Tool] | None = None,
    stop_when: StopCondition | None = None,
    on_step_finish: OnStepFinish | None = None,
    prepare_step: PrepareStep | None = None,
) -> GenerateTextResult:
    """Generate text and optionally execute tools over multiple model steps."""
    if not prompt.strip():
        raise ValueError("A non-empty prompt is required.")

    all_tools = tools or {}
    effective_stop_when = stop_when or step_count_is(1)
    steps: list[StepResult] = []
    continuation = None
    pending_prompt: str | None = prompt
    pending_tool_results: list[ToolResult] = []

    # This emergency ceiling protects our teaching implementation if a custom
    # stop condition is accidentally written so that it can never return True.
    for step_number in range(100):
        active_tools = dict(all_tools)
        if prepare_step is not None:
            prepared = prepare_step(
                PrepareStepOptions(
                    step_number=step_number,
                    steps=list(steps),
                    active_tools=list(active_tools),
                )
            )
            if prepared is not None and prepared.active_tools is not None:
                unknown_names = set(prepared.active_tools) - set(all_tools)
                if unknown_names:
                    raise ValueError(f"prepare_step activated unknown tools: {unknown_names}")
                active_tools = {
                    name: all_tools[name]
                    for name in prepared.active_tools
                }

        model_step = model.do_generate(
            instructions=instructions,
            prompt=pending_prompt,
            tools=active_tools,
            continuation=continuation,
            tool_results=pending_tool_results,
        )
        tool_results = [
            _execute_tool_call(tool_call, active_tools=active_tools)
            for tool_call in model_step.tool_calls
        ]

        step = StepResult(
            step_number=step_number,
            text=model_step.text,
            tool_calls=model_step.tool_calls,
            tool_results=tool_results,
            usage=model_step.usage,
            finish_reason=model_step.finish_reason,
            raw_response=model_step.raw_response,
        )
        steps.append(step)

        if on_step_finish is not None:
            on_step_finish(step)

        if not model_step.tool_calls:
            break
        if effective_stop_when(steps):
            break

        continuation = model_step.continuation
        pending_prompt = None
        pending_tool_results = tool_results
    else:
        raise RuntimeError("Agent loop exceeded the emergency 100-step ceiling.")

    return GenerateTextResult(
        text=steps[-1].text,
        steps=steps,
        total_usage=_sum_usage(steps),
    )


In [ ]:
AGENT_INSTRUCTIONS = """You are a cat health guideline assistant in an agentic RAG lesson.

You have one tool: retrieve_cat_health_guidelines.

For questions about cat health, symptoms, preventive care, nutrition, vaccines, parasites,
dental health, urinary signs, senior cats, stress, behavior, or home monitoring, call the
retrieval tool before answering. Use one focused retrieval query unless another search is
truly necessary.

After retrieval:
- Answer only from the retrieved context.
- Cite source labels such as [Source 1].
- Recommend contacting a veterinarian for medical decisions, urgent symptoms, or worsening symptoms.

For unrelated questions, do not call the tool. Briefly explain that this assistant is scoped
to the cat health guideline corpus.
"""


def print_step(step: StepResult) -> None:
    print(f"\\n--- Step {step.step_number + 1} | finish={step.finish_reason} ---")
    for tool_call in step.tool_calls:
        print(f"Tool call: {tool_call.tool_name}({tool_call.input})")
    for tool_result in step.tool_results:
        preview = json.dumps(tool_result.output, default=str)[:1200]
        print(f"Tool result: {preview}")
    if step.text:
        print(f"Text: {step.text}")


### Single-Step `generate_text()`

The default stop condition is one step. If the model requests retrieval, the application executes the tool and returns the step, but it does not call the model again to turn that result into a final answer.

Inspect `steps[0].tool_calls` and `steps[0].tool_results`. This is useful when an application wants to own the next action itself.


In [ ]:
single_step_result = generate_text(
    model=language_model,
    instructions=AGENT_INSTRUCTIONS,
    prompt="What urinary signs suggest my cat needs urgent veterinary care?",
    tools=tools,
    on_step_finish=print_step,
)

print("\\nFinal text from the one-step call:", repr(single_step_result.text))
print("Steps:", len(single_step_result.steps))


### Multi-Step `generate_text()`

Now pass `stop_when=step_count_is(4)`. After a tool call, the loop can send the tool result back to the model so it can produce a grounded answer.

The loop still ends early when the model returns text without another tool call.


In [ ]:
multi_step_result = generate_text(
    model=language_model,
    instructions=AGENT_INSTRUCTIONS,
    prompt="What urinary signs suggest my cat needs urgent veterinary care?",
    tools=tools,
    stop_when=step_count_is(4),
    on_step_finish=print_step,
)

print("\\nFinal answer:")
print(multi_step_result.text)
print("\\nTotal usage:", multi_step_result.total_usage)


#### Questions

1. What work happened between the first model step and the final model step?
2. Why is the tool result recorded separately from the final answer?
3. What could happen if the loop had no stop condition or emergency ceiling?


## 8. Create a Reusable `ToolLoopAgent`

`generate_text()` is useful for one-off calls. A reusable agent packages the model, instructions, tools, stop condition, callbacks, and optional per-step behavior together.

This small `ToolLoopAgent` is a configured wrapper around the same loop. It does not introduce a new kind of intelligence.


In [ ]:
@dataclass
class ToolLoopAgent:
    model: LanguageModel
    instructions: str
    tools: dict[str, Tool] = field(default_factory=dict)
    stop_when: StopCondition = field(default_factory=lambda: step_count_is(20))
    on_step_finish: OnStepFinish | None = None
    prepare_step: PrepareStep | None = None

    def generate(
        self,
        *,
        prompt: str,
        on_step_finish: OnStepFinish | None = None,
    ) -> GenerateTextResult:
        return generate_text(
            model=self.model,
            instructions=self.instructions,
            prompt=prompt,
            tools=self.tools,
            stop_when=self.stop_when,
            on_step_finish=on_step_finish or self.on_step_finish,
            prepare_step=self.prepare_step,
        )


cat_health_agent = ToolLoopAgent(
    model=language_model,
    instructions=AGENT_INSTRUCTIONS,
    tools=tools,
    stop_when=step_count_is(4),
)


Run a cat-health question and an unrelated question. The first should retrieve. The second should finish without a tool call.


In [ ]:
agent_questions = [
    "What preventive care should a healthy adult cat receive?",
    "Who won the 2022 FIFA World Cup?",
]

for question in agent_questions:
    print("\\nQUESTION:", question)
    result = cat_health_agent.generate(prompt=question, on_step_finish=print_step)
    print("\\nFINAL ANSWER:", result.text)
    print("=" * 100)


### Inspect the Recorded Steps

A plausible final answer can hide a poor path. Inspecting steps reveals whether the model selected the right tool, what query it generated, what the application returned, and how many model calls the run required.


In [ ]:
inspection_result = cat_health_agent.generate(
    prompt="What signs of dental disease should a cat owner watch for?"
)

for step in inspection_result.steps:
    print(
        {
            "step_number": step.step_number,
            "finish_reason": step.finish_reason,
            "tool_calls": [
                {"name": call.tool_name, "input": call.input}
                for call in step.tool_calls
            ],
            "tool_result_count": len(step.tool_results),
            "text_preview": step.text[:200],
        }
    )


## 9. Change Loop Behavior with `prepare_step`

AI SDK's `prepareStep` can change settings before each model step. Our deliberately narrow `prepare_step` can change which tools are active.

The function below disables retrieval after any earlier step requested the retriever. That forces the next model step to answer from the result it already has instead of starting another retrieval step.

This limits **later retrieval steps**, not multiple parallel calls emitted together in one model step. A production tool-call budget would count and constrain individual calls too.


In [ ]:
def disable_retrieval_after_first_retrieval_step(
    options: PrepareStepOptions,
) -> PrepareStepResult | None:
    already_requested_retrieval = any(
        call.tool_name == "retrieve_cat_health_guidelines"
        for step in options.steps
        for call in step.tool_calls
    )
    if already_requested_retrieval:
        return PrepareStepResult(active_tools=[])
    return None


one_retrieval_step_agent = ToolLoopAgent(
    model=language_model,
    instructions=AGENT_INSTRUCTIONS,
    tools=tools,
    stop_when=step_count_is(4),
    prepare_step=disable_retrieval_after_first_retrieval_step,
)

prepared_result = one_retrieval_step_agent.generate(
    prompt=(
        "Summarize urinary emergency signs and annual preventive care. "
        "Use one focused search query."
    ),
    on_step_finish=print_step,
)

print("\\nFinal answer:")
print(prepared_result.text)


#### Questions

1. How is `prepare_step` similar to middleware?
2. Why might disabling tools after retrieval reduce cost and latency?
3. When could that rule reduce answer quality?
4. How does this configured loop compare with the explicit LangGraph model-tools loop in the main notebook?


## Production Library Responsibilities

Our handcrafted agent is intentionally small and useful for learning. A production-ready SDK or agent runtime usually handles much more:

- Provider normalization, provider registries, and model capability differences
- Stateless message-history management and stateful provider continuation strategies
- Retries, rate-limit handling, timeouts, cancellation, and exponential backoff
- Streaming text, tool-call arguments, tool results, and structured events
- Complete JSON-schema validation, tool-call repair, and approval workflows
- Parallel tool execution, per-tool budgets, and durable execution
- Rich stop conditions, dynamic model/tool selection, and loop-state management
- Token accounting, tracing, telemetry, middleware, and error observability
- Persistent vector storage, scalable nearest-neighbor search, and metadata filtering
- Security controls for tools that read data or create side effects

The important part is that these features extend the same loop you implemented here:

```text
model decides -> application executes -> result returns to model -> repeat or answer
```

### Connect Back to the Main Notebook

- LangChain `create_agent` packages a high-level model-tools loop, much like our `ToolLoopAgent`.
- LangChain middleware changes or observes the loop, much like `on_step_finish`, `stop_when`, and `prepare_step`.
- LangGraph makes nodes, state, and conditional edges explicit. Our `generate_text()` loop expresses the same core control flow with a Python loop.
- Qdrant replaces our brute-force in-memory vector search with a purpose-built vector database.
